# Invesco HTML Scraping Sandbox

Replacing the dead `dng-api.invesco.com` JSON endpoint with Playwright HTML scraping.

**PCY page:** `https://www.invesco.com/us/en/financial-products/etfs/invesco-emerging-markets-sovereign-debt-etf.html`

In [ ]:
import sys
sys.path.insert(0, '..')

import re
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

## Step 1 — Fetch with Playwright

In [ ]:
import json

PCY_URL = "https://www.invesco.com/us/en/financial-products/etfs/invesco-emerging-markets-sovereign-debt-etf.html"

captured = {}  # url_key -> json payload

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    page = await browser.new_page()

    async def capture(route, request):
        resp = await route.fetch()
        url = request.url
        if "holding" in url.lower() or "weightedHolding" in url:
            try:
                key = url.split("?")[0].split("/")[-2] + "/" + url.split("?")[0].split("/")[-1]
                params = dict(p.split("=") for p in url.split("?")[-1].split("&") if "=" in p)
                full_key = key + ("/" + params.get("breakdown", "") if "breakdown" in params else "")
                captured[full_key] = await resp.json()
            except Exception:
                pass
        await route.fulfill(response=resp)

    await page.route("**/*", capture)
    await page.goto(PCY_URL, wait_until="networkidle", timeout=60_000)
    html = await page.content()
    await browser.close()

print("Captured endpoints:")
for k, v in captured.items():
    top = {key: val for key, val in v.items() if key != "holdings"}
    print(f"  {k}: {top}")
    if "holdings" in v:
        print(f"    holdings[0]: {v['holdings'][0] if v['holdings'] else 'empty'}")

## Step 2 — Inspect the HTML structure

Find anything containing Duration / Yield / Spread / Maturity.

In [ ]:
soup = BeautifulSoup(html, "html.parser")

# All elements whose text matches fund characteristic keywords
for el in soup.find_all(string=re.compile(r"Duration|Yield|Spread|Maturity|Coupon", re.I)):
    parent = el.parent
    print(f"{parent.get('class')}  →  {el.strip()!r}")

In [ ]:
# Broad sweep of tabular-list elements
for el in soup.find_all(class_=re.compile(r"tabular-list")):
    text = el.get_text(separator=" | ", strip=True)
    if text:
        print(f"{el.get('class')}  →  {text[:120]}")

## Step 3 — Extract label → value pairs

Adjust selectors based on what Step 2 shows.

In [ ]:
def parse_invesco_characteristics(html: str) -> dict[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    result = {}
    for row in soup.find_all(class_="tabular-list__list"):
        label_el = row.find(class_="tabular-list__label")
        value_el = row.find(class_="tabular-list__value")
        if not label_el or not value_el:
            continue
        label = re.sub(r"\s*\(as of [^)]+\)", "", label_el.get_text(strip=True))
        value = value_el.get_text(strip=True)
        result[label] = value
    return result

# Debug: confirm selector hits before parsing
soup_debug = BeautifulSoup(html, "html.parser")
hits = soup_debug.find_all(class_="tabular-list__list")
print(f"find_all('tabular-list__list') → {len(hits)} elements")
if hits:
    first = hits[0]
    print("first label:", first.find(class_="tabular-list__label"))
    print("first value:", first.find(class_="tabular-list__value"))

chars = parse_invesco_characteristics(html)
print(f"\nParsed {len(chars)} entries:")
for k, v in chars.items():
    print(f"  {k!r}: {v!r}")

## Step 4 — Map to ETFAnalytics fields

Update `LABEL_MAP` keys with the exact strings printed in Step 3.

In [ ]:
def _parse_number(value: str) -> float | None:
    """Strip units (yrs, %, $) and return float. Returns None for '--'."""
    if not value or value.strip() in ("--", "-", ""):
        return None
    m = re.search(r"[-+]?[\d,]+\.?\d*", value.replace(",", ""))
    return float(m.group()) if m else None

# Exact label strings from the page (lowercase, date suffix stripped)
LABEL_MAP = {
    "Effective duration":   "effective_duration",
    "Modified duration":    "modified_duration",
    "Yield to maturity":    "ytm",
    "Yield to worst":       "ytw",
    "Years to maturity":    "avg_maturity",   # no OAS available on this page
    "Weighted avg coupon":  "avg_coupon",
}

parsed = {field: _parse_number(chars[label]) for label, field in LABEL_MAP.items() if label in chars}
print(parsed)

# Verify all expected fields are present
missing = [label for label in LABEL_MAP if label not in chars]
if missing:
    print(f"MISSING labels: {missing}")

In [ ]:
soup_h = BeautifulSoup(html, "html.parser")

# 1. Look for any download/CSV links
print("=== Download / CSV links ===")
for a in soup_h.find_all("a", href=True):
    href = a["href"]
    text = a.get_text(strip=True)
    if any(kw in href.lower() or kw in text.lower() for kw in ("holding", "csv", "download", "xls", "excel")):
        print(f"  {text!r}  →  {href}")

# 2. Look for holdings table rows
print("\n=== Holdings table elements ===")
for el in soup_h.find_all(class_=re.compile(r"holding", re.I)):
    text = el.get_text(separator=" | ", strip=True)
    if text:
        print(f"  {el.get('class')}  →  {text[:120]}")

# 3. Any JSON blobs embedded in the page (sometimes holdings are inline)
print("\n=== JSON data islands (holdings-related) ===")
for script in soup_h.find_all("script", type=re.compile(r"json", re.I)):
    text = script.string or ""
    if any(kw in text.lower() for kw in ("holding", "cusip", "coupon", "maturity")):
        print(text[:300])

## Step 5 — Find holdings data